In [53]:
from typing import TypedDict
from langgraph.graph import StateGraph,START,END

In [54]:
class SongState(TypedDict):
    user_input: str
    song:str
    response: str

In [55]:
PREFIX = "Recommend songs like "

def identify_song(state: SongState) -> dict:
    request = state["user_input"]
    if not request.startswith(PREFIX):
        raise ValueError(f"Request must start with {PREFIX!r}")
    song_name = request.removeprefix(PREFIX).strip()
    if not song_name:
        raise ValueError("Song name cannot be empty")
    return {"song": song_name}

def generate_response(state: SongState) -> dict:
    return {"response": f"You selected {state['song']}. The recommendation engine can run next."}

In [56]:
builder = StateGraph(SongState)
builder.add_node("identify_song", identify_song)
builder.add_node("generate_response", generate_response)
builder.add_edge(START, "identify_song")
builder.add_edge("identify_song", "generate_response")
builder.add_edge("generate_response", END)
graph = builder.compile()
print("Registered nodes:", list(builder.nodes))

Registered nodes: ['identify_song', 'generate_response']


In [57]:
initial_state = {"user_input": "Recommend songs like Comedy", "song": "", "response": ""}
result = graph.invoke(initial_state)
print(result)

{'user_input': 'Recommend songs like Comedy', 'song': 'Comedy', 'response': 'You selected Comedy. The recommendation engine can run next.'}


In [58]:
print("UPDATES")
for event in graph.stream(initial_state, stream_mode="updates"):
    print(event)

print("\nVALUES")
for state in graph.stream(initial_state, stream_mode="values"):
    print(state)

UPDATES
{'identify_song': {'song': 'Comedy'}}
{'generate_response': {'response': 'You selected Comedy. The recommendation engine can run next.'}}

VALUES
{'user_input': 'Recommend songs like Comedy', 'song': '', 'response': ''}
{'user_input': 'Recommend songs like Comedy', 'song': 'Comedy', 'response': ''}
{'user_input': 'Recommend songs like Comedy', 'song': 'Comedy', 'response': 'You selected Comedy. The recommendation engine can run next.'}


In [59]:
test_cases = {
    "Recommend songs like Comedy": "Comedy",
    "Recommend songs like Blinding Lights": "Blinding Lights",
}

for request, expected_song in test_cases.items():
    result = graph.invoke({"user_input": request, "song": "", "response": ""})
    assert result["song"] == expected_song, (request, result)
    assert expected_song in result["response"]
    print(f"PASS: {request} -> {result['song']}")

print("All test cases passed")

PASS: Recommend songs like Comedy -> Comedy
PASS: Recommend songs like Blinding Lights -> Blinding Lights
All test cases passed
